In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

import logging

In [2]:
# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

In [3]:
# Configuración del modelo de lenguaje
load_dotenv()

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)


### Fundamentos de RAG

| Estrategia | Caracteristicas | Limitaciones |
|--|--|--|
| Contexto | Incluir todo la información de los documentos en el PROMPT | Ventana del contexto |
| Fine tuning | Reentranar el modelo, atado a la infraestructura | Tiempo de actualización |
| RAG | Particiona el docuemnto en fragmentos y coloca los fragmentos en el PROMPT | Tratamiento de los documentos |


In [ ]:
# Consulta sin tener la informacion de la politica de devolucion de la empresa TiendaYa

PREGUNTA = "¿Cuál es la política de devolución de la empresa TiendaYa para productos en oferta?"

respuesta = llm.invoke(PREGUNTA)

print("Respuesta: " + respuesta.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Respuesta: Como modelo de inteligencia artificial, no tengo acceso a información en tiempo real ni a bases de datos internas de empresas específicas llamadas "TiendaYa" (ya que puede haber varios comercios con ese nombre en diferentes países). 

Para conocer la política de devolución exacta para productos en oferta de **TiendaYa**, te recomiendo hacer lo siguiente:

1. **Revisar su sitio web oficial:** Busca en el pie de página (footer) secciones como *"Términos y condiciones"*, *"Políticas de devolución"* o *"Preguntas frecuentes (FAQ)"*.
2. **Contactar a su servicio al cliente:** Escríbeles a través de su chat en vivo, correo electrónico o WhatsApp oficial si lo tienen disponible.

*Nota general sobre el comercio electrónico:* Muchas tiendas online suelen tener políticas más restrictivas con los productos en oferta o liquidación (por ejemplo, permitiendo solo cambios por talla/defecto y no devoluciones de dinero), por lo que siempre es mejor confirmarlo directamente con ellos antes d

### Aplicando RAG

- 1. Carga y procesamiento del documento. ( INGESTA DE DATOS)
- 2. Realizar el embedding usando modelos de IA. ( INGESTA DE DATOS)
- 3. Uso de base de datos vectorial ( INGESTA DE DATOS)
- 4. Realización de consultas

#### 1. Carga y procesamiento del documento

In [7]:
# Librería pypdf

from pypdf import PdfReader
from langchain_core.documents import Document

ARCHIVO = "data/politica_tiendaya.pdf"

reader = PdfReader(ARCHIVO)

documentos = []

for i, pagina in enumerate(reader.pages):
    
    texto = pagina.extract_text()
    print(f"Página {i+1}: {texto[:100]}...")  # Muestra los primeros 100 caracteres de cada página

    doc = Document(page_content=texto, 
                   metadata={"pagina": i+1})
    
    documentos.append(doc)

print("Número de páginas procesadas: ", len(documentos))


Página 1: TiendaYa — Política de devoluciones y reembolsos
Versión 2.1 · Documento ficticio para uso en el cur...
Página 2: 5. Productos dañados en el envío
Si el producto llega dañado, el cliente debe reportarlo dentro de l...
Página 3: 8. Cómo iniciar una devolución
El cliente debe ingresar a la sección Mis pedidos, seleccionar el ped...
Número de páginas procesadas:  3


In [12]:
# Segmentacion de documentos

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Se va a segmentar el documento en fragmetos con solapamientos

# Dividir el fragmentos de 500 carateres con un solapamiento de 50 caracteres

chunk_size = 500
chunk_overlap = 50

spliter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,  # fragmento de 500 caracteres
    chunk_overlap=chunk_overlap # solapamiento de 50 caracteres
)

fragmentos = spliter.split_documents(documentos)

print(f"{'chunk_size':>10} {'overlap':>8} {'fragmentos':>11}")
print(f"{chunk_size:>10} {chunk_overlap:>8} {len(fragmentos):>11}")

chunk_size  overlap  fragmentos
       500       50           7
